# Experiment results

Load the per-question diagnostic output, summarize it with pandas, and plot recall and answer accuracy. Change `SPLIT` or the selected run paths to inspect another run. The bootstrap interval resamples question IDs so comparisons use the same sampled questions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
SPLIT = 'dev'
diagnostic_dir = ROOT / 'data/experiments/diagnostic' / SPLIT
diagnostic_run = max(diagnostic_dir.glob('*/per_question.csv'), key=lambda p: int(p.parent.name))
df = pd.read_csv(diagnostic_run)
summary = df.groupby(['chunking', 'method']).mean(numeric_only=True)
summary

In [ ]:
recall = summary.reset_index().pivot_table(index='chunking', columns='method', values='recall_medium')
recall.plot.bar(title='Medium-budget retrieval recall', ylabel='Recall', ylim=(0, 1), figsize=(9, 4))
plt.tight_layout()
plt.show()

In [ ]:
# Paired percentile bootstrap: each resample contains the same question IDs for every setup.
rng = np.random.default_rng(42)
groups = df.groupby(['chunking', 'method'])
ci = groups['recall_medium'].mean().to_frame('mean')
boot = {key: [] for key in groups.groups}
question_ids = df.question_id.unique()
for _ in range(2000):
    sampled = rng.choice(question_ids, len(question_ids), replace=True)
    sample = pd.concat([df[df.question_id == q] for q in sampled])
    for key, value in sample.groupby(['chunking', 'method'])['recall_medium'].mean().items():
        boot[key].append(value)
ci['ci_low'] = [np.quantile(boot[key], .025) for key in ci.index]
ci['ci_high'] = [np.quantile(boot[key], .975) for key in ci.index]
ci

In [ ]:
llm_dir = ROOT / 'data/experiments/llm' / SPLIT
llm_runs = list(llm_dir.glob('*/per_question.csv'))
if llm_runs:
    llm_run = max(llm_runs, key=lambda p: int(p.parent.name))
    answers = pd.read_csv(llm_run)
    accuracy = answers.groupby('configuration')['correct'].mean().sort_values()
    accuracy.plot.bar(title='LLM answer accuracy', ylabel='Accuracy', ylim=(0, 1), figsize=(12, 4))
    plt.tight_layout()
    plt.show()
else:
    print(f'No LLM run found for {SPLIT}.')